# Sparse lookup and legacy QC

Summarizes the versioned sparse index and quantifies known historical Site-II/Site-III overhang mismatches.

**Rules:** `legacy-optimized-v1`; **seed:** 42 unless explicitly noted.

In [ ]:
REPO = '/home/wendai/projects/hurdler/clone_repeat_protein'
RULE_PROFILE = 'legacy-optimized-v1'
INDEX_DIR = '/net/scratch/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step01_reference_lookup/runs/run01_production/raw/legacy-optimized-v1'
LEGACY_OUTPUT_DIR = '/home/wendai/projects/hurdler/clone_repeat_protein/output'
QC_OUTPUT = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step01_reference_lookup/tables/legacy_qc.json'

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd

def sha256(path):
    path = Path(path)
    if not path.is_file(): return None
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

run_context = {'rule_profile': RULE_PROFILE, 'input_hashes': {}, 'row_counts': {}, 'filter_flow': [], 'limitations': []}

In [ ]:
from hurdler.index import PatternIndex
from hurdler.qc import legacy_qc
index = PatternIndex.load(INDEX_DIR)
qc = legacy_qc(LEGACY_OUTPUT_DIR, INDEX_DIR, QC_OUTPUT)
run_context['input_hashes']['pattern_index.npz'] = sha256(Path(INDEX_DIR) / 'pattern_index.npz')
run_context['row_counts'] = {'patterns': len(index.keys), 'enzyme_pairs': len(index.pair_table)}
run_context['limitations'] = [qc['note']]
pd.DataFrame([qc])

In [ ]:
index.pair_table.groupby(['site_i_ovhg', 'site_ii_ovhg']).size().rename('enzyme_pairs').reset_index()